# 05_parameter_estimation_realistic

This notebook converts the shared cleaned datasets and the corrected Part 1 demand table into model-ready inputs for the realistic childcare expansion model.

Main tasks:
1. load the shared cleaned datasets and corrected benchmark demand table
2. construct realistic facility-level expansion parameters
3. build candidate-site and distance-conflict tables
4. export all realistic-model input tables


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
# Display options
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

In [3]:
# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SHARED_DIR = PROJECT_ROOT / "data" / "processed" / "shared"
IDEAL_DIR = PROJECT_ROOT / "data" / "processed" / "ideal"
REALISTIC_DIR = PROJECT_ROOT / "data" / "processed" / "realistic"

REALISTIC_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SHARED_DIR:", SHARED_DIR)
print("IDEAL_DIR:", IDEAL_DIR)
print("REALISTIC_DIR:", REALISTIC_DIR)

PROJECT_ROOT: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project
SHARED_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/shared
IDEAL_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/ideal
REALISTIC_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/realistic


In [4]:
# Check required input files
required_files = [
    SHARED_DIR / "clean_childcare_facilities.csv",
    SHARED_DIR / "facility_geo_ready.csv",
    SHARED_DIR / "potential_locations_geo_ready.csv",
    IDEAL_DIR / "zipcode_demand_supply_ideal.csv",
    IDEAL_DIR / "build_options_ideal.csv",
]

missing_files = [str(f) for f in required_files if not f.exists()]

print("Required input files check:")
for path in required_files:
    print(path.name, "exists ->", path.exists())

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

Required input files check:
clean_childcare_facilities.csv exists -> True
facility_geo_ready.csv exists -> True
potential_locations_geo_ready.csv exists -> True
zipcode_demand_supply_ideal.csv exists -> True
build_options_ideal.csv exists -> True


In [5]:
# Load shared and ideal inputs
fac_clean = pd.read_csv(SHARED_DIR / "clean_childcare_facilities.csv")
facility_geo = pd.read_csv(SHARED_DIR / "facility_geo_ready.csv")
candidate_geo = pd.read_csv(SHARED_DIR / "potential_locations_geo_ready.csv")
zip_df = pd.read_csv(IDEAL_DIR / "zipcode_demand_supply_ideal.csv")
build_options = pd.read_csv(IDEAL_DIR / "build_options_ideal.csv")

In [6]:
# Quick preview
datasets = {
    "fac_clean": fac_clean,
    "facility_geo": facility_geo,
    "candidate_geo": candidate_geo,
    "zip_df": zip_df,
    "build_options": build_options,
}

for name, df in datasets.items():
    print(f"\n{name}: {df.shape}")
    display(df.head())


fac_clean: (7740, 20)


,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,is_active_facility,geo_missing_flag,under5_capacity_current,capacity_inconsistency_flag,zero_total_capacity_flag
0,51349,FDC,Registration,"Valerio, Gladys",Bronx,10474,Bronx 8,0,0,0,0,6,6,40.818399,-73.888816,1,False,6,0,0
1,73544,SACC,Registration,YMCA OF GREATER NY,New York,10017,Manhattan 2,0,0,0,60,0,60,40.753216,-73.970794,1,False,0,0,0
2,108407,GFDC,License,"Almond Tree Group Family Day Care, LLC",Brooklyn,11225,Brooklyn 17,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0
3,111953,GFDC,License,"Williams, Petal",Brooklyn,11226,Brooklyn 22,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0
4,126425,GFDC,License,"Hernandez, Lidia",Bronx,10465,Bronx 8,0,0,0,0,12,12,40.841228,-73.823572,1,False,12,0,0



facility_geo: (7740, 16)


,facility_id,facility_name,program_type,facility_status,is_active_facility,zipcode,latitude,longitude,geo_missing_flag,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,under5_capacity_current,total_capacity
0,51349,"Valerio, Gladys",FDC,Registration,1,10474,40.818399,-73.888816,False,0,0,0,0,6,6,6
1,73544,YMCA OF GREATER NY,SACC,Registration,1,10017,40.753216,-73.970794,False,0,0,0,60,0,0,60
2,108407,"Almond Tree Group Family Day Care, LLC",GFDC,License,1,11225,NaN,NaN,True,0,0,0,4,12,12,16
3,111953,"Williams, Petal",GFDC,License,1,11226,NaN,NaN,True,0,0,0,4,12,12,16
4,126425,"Hernandez, Lidia",GFDC,License,1,10465,40.841228,-73.823572,False,0,0,0,0,12,12,12



candidate_geo: (31100, 5)


,candidate_id,zipcode,latitude,longitude,geo_missing_flag
0,1,10001,40.741893,-74.000140,False
1,2,10001,40.752007,-74.005436,False
2,3,10001,40.750545,-73.997147,False
3,4,10001,40.744080,-74.001932,False
4,5,10001,40.748690,-73.999341,False



zip_df: (311, 23)


,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609.0,24.0,9.0,58.0,472.0,1,1,1
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724.0,198.0,56.0,0.0,1230.0,0,1,1
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995.0,0.0,7.0,0.0,960.0,0,1,1
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263.0,0.0,2.0,0.0,289.0,0,1,1
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39.0,0.0,1.0,363.0,323.0,1,1,1



build_options: (3, 4)


,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost
0,small,100,50,65000
1,medium,200,100,95000
2,large,400,200,115000


## Demand adjustments

The realistic branch inherits the benchmark demand table and preserves the upstream correction that zero-child ZIP codes should have zero total-capacity requirement.


In [7]:
# Standardize zipcode and identifier columns
def standardize_zipcode(series):
    return (
        series.astype(str)
              .str.strip()
              .str.replace(r"\.0$", "", regex=True)
              .str.zfill(5)
    )

for df in [fac_clean, facility_geo, candidate_geo, zip_df]:
    if "zipcode" in df.columns:
        df["zipcode"] = standardize_zipcode(df["zipcode"])

fac_clean["facility_id"] = fac_clean["facility_id"].astype(str)
facility_geo["facility_id"] = facility_geo["facility_id"].astype(str)
candidate_geo["candidate_id"] = pd.to_numeric(candidate_geo["candidate_id"], errors="coerce").astype("Int64")

# Repair zero-child zipcode artifacts inherited from Part 1
zipcode_demand_supply_realistic = zip_df.copy()
zero_child_mask = zipcode_demand_supply_realistic["child_pop_0_12"].fillna(0) <= 0
zero_child_req_total_fixed = int((zero_child_mask & (zipcode_demand_supply_realistic["req_total"].fillna(0) > 0)).sum())

zipcode_demand_supply_realistic.loc[zero_child_mask, "req_total"] = 0
zipcode_demand_supply_realistic["gap_total"] = (
    zipcode_demand_supply_realistic["req_total"] - zipcode_demand_supply_realistic["existing_total"]
).clip(lower=0)
zipcode_demand_supply_realistic["is_desert_total_before"] = (
    zipcode_demand_supply_realistic["gap_total"] > 0
).astype(int)
zipcode_demand_supply_realistic["needs_intervention_before"] = (
    (zipcode_demand_supply_realistic["gap_total"] > 0)
    | (zipcode_demand_supply_realistic["gap_under5"] > 0)
).astype(int)

print("Zero-child zipcodes:", int(zero_child_mask.sum()))
print("Zero-child zipcodes with req_total reset:", zero_child_req_total_fixed)
display(zipcode_demand_supply_realistic.loc[zero_child_mask, [
    "zipcode", "child_pop_0_12", "req_total", "req_under5", "existing_total", "existing_under5"
]].head(20))

Zero-child zipcodes: 131
Zero-child zipcodes with req_total reset: 0


,zipcode,child_pop_0_12,req_total,req_under5,existing_total,existing_under5
7,10008,0.0,0,0,0.0,0.0
18,10020,0.0,0,0,44.0,0.0
39,10041,0.0,0,0,0.0,0.0
40,10043,0.0,0,0,0.0,0.0
42,10045,0.0,0,0,0.0,0.0
43,10055,0.0,0,0,0.0,0.0
44,10060,0.0,0,0,0.0,0.0
48,10080,0.0,0,0,0.0,0.0
49,10081,0.0,0,0,0.0,0.0
50,10087,0.0,0,0,0.0,0.0


## Existing facility expansion parameters

These cells rebuild the facility-side expansion logic for Part 2 by applying the stricter 20\% cap and the three realistic expansion bands.


In [8]:
# Filter active facilities and construct realistic expansion parameters
fac_existing = fac_clean.copy()
fac_existing = fac_existing[fac_existing["is_active_facility"] == 1].copy()
fac_existing = fac_existing[fac_existing["total_capacity"].fillna(0) > 0].copy()

fac_existing["n_f"] = pd.to_numeric(fac_existing["total_capacity"], errors="coerce").fillna(0).astype(int)
fac_existing["b_f"] = pd.to_numeric(fac_existing["under5_capacity_current"], errors="coerce").fillna(0).astype(int)

geo_cols = ["facility_id", "latitude", "longitude", "zipcode"]
fac_existing = fac_existing.merge(
    facility_geo[geo_cols],
    on="facility_id",
    how="left",
    suffixes=("", "_geo"),
)
fac_existing["zipcode"] = fac_existing["zipcode"].fillna(fac_existing["zipcode_geo"])
fac_existing = fac_existing.drop(columns=["zipcode_geo"], errors="ignore")

# Use cumulative floors so the three blocks add to floor(0.20 * n_f)
fac_existing["cap10_cum"] = np.floor(0.10 * fac_existing["n_f"]).astype(int)
fac_existing["cap15_cum"] = np.floor(0.15 * fac_existing["n_f"]).astype(int)
fac_existing["cap20_cum"] = np.floor(0.20 * fac_existing["n_f"]).astype(int)

fac_existing["cap1"] = fac_existing["cap10_cum"]
fac_existing["cap2"] = (fac_existing["cap15_cum"] - fac_existing["cap10_cum"]).clip(lower=0).astype(int)
fac_existing["cap3"] = (fac_existing["cap20_cum"] - fac_existing["cap15_cum"]).clip(lower=0).astype(int)
fac_existing["U_f_realistic"] = fac_existing["cap20_cum"]

fac_existing["coef1"] = (20000 + 200 * fac_existing["n_f"]) / fac_existing["n_f"]
fac_existing["coef2"] = (20000 + 400 * fac_existing["n_f"]) / fac_existing["n_f"]
fac_existing["coef3"] = (20000 + 1000 * fac_existing["n_f"]) / fac_existing["n_f"]

facility_expansion_params_realistic = fac_existing[[
    "facility_id",
    "facility_name",
    "program_type",
    "zipcode",
    "latitude",
    "longitude",
    "n_f",
    "b_f",
    "cap1",
    "cap2",
    "cap3",
    "cap10_cum",
    "cap15_cum",
    "cap20_cum",
    "U_f_realistic",
    "coef1",
    "coef2",
    "coef3",
]].copy()

display(facility_expansion_params_realistic.head())

,facility_id,facility_name,program_type,zipcode,latitude,longitude,n_f,b_f,cap1,cap2,cap3,cap10_cum,cap15_cum,cap20_cum,U_f_realistic,coef1,coef2,coef3
0,51349,"Valerio, Gladys",FDC,10474,40.818399,-73.888816,6,6,0,0,1,0,0,1,1,3533.333333,3733.333333,4333.333333
1,73544,YMCA OF GREATER NY,SACC,10017,40.753216,-73.970794,60,0,6,3,3,6,9,12,12,533.333333,733.333333,1333.333333
2,108407,"Almond Tree Group Family Day Care, LLC",GFDC,11225,NaN,NaN,16,12,1,1,1,1,2,3,3,1450.000000,1650.000000,2250.000000
3,111953,"Williams, Petal",GFDC,11226,NaN,NaN,16,12,1,1,1,1,2,3,3,1450.000000,1650.000000,2250.000000
4,126425,"Hernandez, Lidia",GFDC,10465,40.841228,-73.823572,12,12,1,0,1,1,1,2,2,1866.666667,2066.666667,2666.666667


In [9]:
# Facility parameter QA checks
print("Any negative block capacities?", (facility_expansion_params_realistic[["cap1", "cap2", "cap3"]] < 0).any().any())
print("Any missing cost coefficients?", facility_expansion_params_realistic[["coef1", "coef2", "coef3"]].isna().any().any())
print("Citywide realistic expansion capacity:", int(facility_expansion_params_realistic["U_f_realistic"].sum()))
print("Existing facilities missing coordinates:", int(facility_expansion_params_realistic["latitude"].isna().sum()))
display(facility_expansion_params_realistic[["n_f", "cap1", "cap2", "cap3", "U_f_realistic"]].describe())

Any negative block capacities? False
Any missing cost coefficients? False
Citywide realistic expansion capacity: 61197
Existing facilities missing coordinates: 169


,n_f,cap1,cap2,cap3,U_f_realistic
count,7712.000000,7712.000000,7712.000000,7712.000000,7712.000000
mean,41.169735,3.603216,2.185944,2.146136,7.935296
std,71.050338,7.176778,3.503844,3.572404,14.223792
min,3.000000,0.000000,0.000000,0.000000,0.000000
25%,14.000000,1.000000,1.000000,1.000000,2.000000
50%,16.000000,1.000000,1.000000,1.000000,3.000000
75%,16.000000,1.000000,1.000000,1.000000,3.000000
max,890.000000,89.000000,44.000000,45.000000,178.000000


## Assignment-cost schematic

The next figure visualizes the original assignment Part 2 expansion-cost rule using the assignment-style example of a facility with $n_f = 100$ slots. Dashed vertical segments mark the threshold jumps at 10\% and 15\%; the final cap is at 20\% of current capacity.


In [ ]:
# Plot the original assignment Part 2 cost formula for n_f = 100 and export a report-ready figure
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator

REPORT_FIG_DIR = PROJECT_ROOT / "report" / "figures"
REPORT_FIG_DIR.mkdir(parents=True, exist_ok=True)

n = 100
cut1 = int(np.floor(0.10 * n))
cut2 = int(np.floor(0.15 * n))
cut3 = int(np.floor(0.20 * n))
coef1 = (20000 + 200 * n) / n
coef2 = (20000 + 400 * n) / n
coef3 = (20000 + 1000 * n) / n

x1 = np.linspace(0, cut1, 200)
y1 = coef1 * x1
x2 = np.linspace(cut1, cut2, 200)
y2 = coef2 * x2
x3 = np.linspace(cut2, cut3, 200)
y3 = coef3 * x3

jump1_low, jump1_high = coef1 * cut1, coef2 * cut1
jump2_low, jump2_high = coef2 * cut2, coef3 * cut2

fig, ax = plt.subplots(figsize=(6.6, 4.2), constrained_layout=True)

band_colors = ["#dceeff", "#e6f4df", "#fff0d8"]
curve_color = "#173b78"

ax.axvspan(0, cut1, color=band_colors[0], alpha=0.65)
ax.axvspan(cut1, cut2, color=band_colors[1], alpha=0.65)
ax.axvspan(cut2, cut3, color=band_colors[2], alpha=0.65)

ax.plot(x1, y1, color=curve_color, linewidth=2.8)
ax.plot(x2, y2, color=curve_color, linewidth=2.8)
ax.plot(x3, y3, color=curve_color, linewidth=2.8)

ax.vlines([cut1, cut2], [jump1_low, jump2_low], [jump1_high, jump2_high],
          colors=curve_color, linestyles=(0, (4, 3)), linewidth=1.6, alpha=0.85)

ax.scatter([cut1, cut2, cut3], [jump1_low, jump2_low, coef3 * cut3],
           color=curve_color, s=34, zorder=5)
ax.scatter([cut1, cut2], [jump1_high, jump2_high],
           facecolors='white', edgecolors=curve_color, linewidths=1.8, s=74, zorder=6)

ax.set_title(r'Original assignment cost rule for $n_f = 100$')
ax.set_xlabel('Added slots $x_f$')
ax.set_ylabel('Total expansion cost')
ax.set_xlim(0, cut3)
ax.set_ylim(0, coef3 * cut3 * 1.06)
ax.set_xticks([0, cut1, cut2, cut3])
ax.yaxis.set_major_locator(MultipleLocator(5000))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: '$0' if x == 0 else f'${x/1000:.0f}k'))
ax.grid(axis='y', alpha=0.18)
ax.grid(axis='x', alpha=0.08)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.text(cut1 / 2, 0.97, '0-10%\n$400/slot$', transform=ax.get_xaxis_transform(),
        ha='center', va='top', fontsize=9)
ax.text((cut1 + cut2) / 2, 0.97, '10-15%\n$600/slot$', transform=ax.get_xaxis_transform(),
        ha='center', va='top', fontsize=9)
ax.text((cut2 + cut3) / 2, 0.97, '15-20%\n$1200/slot$', transform=ax.get_xaxis_transform(),
        ha='center', va='top', fontsize=9)

fig_path = REPORT_FIG_DIR / "realistic_expansion_functions.png"
fig.savefig(fig_path, dpi=220, facecolor='white')
plt.show()
print("Saved:", fig_path)


## Candidate-site build options

Candidate locations are paired with the same discrete build menu used in Part 1 so the realistic model can make site-specific new-facility decisions.


In [10]:
# Candidate-site and build-option setup
candidate_required_cols = ["candidate_id", "zipcode", "latitude", "longitude"]
build_required_cols = ["size", "new_total_capacity", "new_under5_capacity_max", "fixed_build_cost"]

missing_candidate_cols = [c for c in candidate_required_cols if c not in candidate_geo.columns]
missing_build_cols = [c for c in build_required_cols if c not in build_options.columns]

if missing_candidate_cols:
    raise KeyError(f"Missing columns in potential_locations_geo_ready.csv: {missing_candidate_cols}")
if missing_build_cols:
    raise KeyError(f"Missing columns in build_options_ideal.csv: {missing_build_cols}")

candidate_geo = candidate_geo.copy()
candidate_geo["zipcode"] = standardize_zipcode(candidate_geo["zipcode"])

candidate_build_options_realistic = (
    candidate_geo.assign(key=1)
    .merge(build_options.assign(key=1), on="key", how="inner")
    .drop(columns=["key"])
)

print("Candidate sites:", candidate_geo["candidate_id"].nunique())
print("Candidate-size options:", len(candidate_build_options_realistic))
display(candidate_build_options_realistic.head())

Candidate sites: 31100
Candidate-size options: 93300


,candidate_id,zipcode,latitude,longitude,geo_missing_flag,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost
0,1,10001,40.741893,-74.000140,False,small,100,50,65000
1,1,10001,40.741893,-74.000140,False,medium,200,100,95000
2,1,10001,40.741893,-74.000140,False,large,400,200,115000
3,2,10001,40.752007,-74.005436,False,small,100,50,65000
4,2,10001,40.752007,-74.005436,False,medium,200,100,95000


In [11]:
# Helper for distance calculations
def haversine_miles(lat1, lon1, lat2, lon2):
    radius_miles = 3958.7613

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))
    return radius_miles * c

MIN_DIST = 0.06

## Distance conflict tables

This section computes the candidate-existing and candidate-candidate spacing conflicts that define the feasible siting set in the realistic model.


In [12]:
# Candidate-existing conflicts
cand_exist_pairs = candidate_geo.merge(
    facility_expansion_params_realistic[["facility_id", "zipcode", "latitude", "longitude"]],
    on="zipcode",
    how="inner",
    suffixes=("_cand", "_exist"),
)

cand_exist_pairs["distance_miles"] = haversine_miles(
    cand_exist_pairs["latitude_cand"],
    cand_exist_pairs["longitude_cand"],
    cand_exist_pairs["latitude_exist"],
    cand_exist_pairs["longitude_exist"],
)

candidate_existing_conflicts = cand_exist_pairs[
    cand_exist_pairs["distance_miles"] < MIN_DIST
][["candidate_id", "facility_id", "zipcode", "distance_miles"]].drop_duplicates()

print("Candidate-existing conflicts:", candidate_existing_conflicts.shape)
display(candidate_existing_conflicts.head())

Candidate-existing conflicts: (4730, 4)


,candidate_id,facility_id,zipcode,distance_miles
36,5,837597,10001,0.026513
41,5,229433,10001,0.026513
423,48,837597,10001,0.021816
428,48,229433,10001,0.021816
585,66,837597,10001,0.043996


In [13]:
# Candidate-candidate conflicts
cand1 = candidate_geo.rename(columns={
    "candidate_id": "candidate_id_1",
    "latitude": "latitude_1",
    "longitude": "longitude_1",
})
cand2 = candidate_geo.rename(columns={
    "candidate_id": "candidate_id_2",
    "latitude": "latitude_2",
    "longitude": "longitude_2",
})

candidate_candidate_pairs = cand1.merge(cand2, on="zipcode", how="inner")
candidate_candidate_pairs = candidate_candidate_pairs[
    candidate_candidate_pairs["candidate_id_1"] < candidate_candidate_pairs["candidate_id_2"]
].copy()

candidate_candidate_pairs["distance_miles"] = haversine_miles(
    candidate_candidate_pairs["latitude_1"],
    candidate_candidate_pairs["longitude_1"],
    candidate_candidate_pairs["latitude_2"],
    candidate_candidate_pairs["longitude_2"],
)

candidate_candidate_conflicts = candidate_candidate_pairs[
    candidate_candidate_pairs["distance_miles"] < MIN_DIST
][["candidate_id_1", "candidate_id_2", "zipcode", "distance_miles"]].drop_duplicates()

print("Candidate-candidate conflicts:", candidate_candidate_conflicts.shape)
display(candidate_candidate_conflicts.head())

Candidate-candidate conflicts: (11455, 4)


,candidate_id_1,candidate_id_2,zipcode,distance_miles
446,5,47,10001,0.047333
447,5,48,10001,0.013688
465,5,66,10001,0.020589
581,6,82,10001,0.016552
644,7,45,10001,0.035234


In [14]:
# Existing-existing distance diagnostics
exist1 = facility_expansion_params_realistic.rename(columns={
    "facility_id": "facility_id_1",
    "latitude": "latitude_1",
    "longitude": "longitude_1",
})
exist2 = facility_expansion_params_realistic.rename(columns={
    "facility_id": "facility_id_2",
    "latitude": "latitude_2",
    "longitude": "longitude_2",
})

existing_existing_pairs = exist1.merge(exist2, on="zipcode", how="inner")
existing_existing_pairs = existing_existing_pairs[
    existing_existing_pairs["facility_id_1"] < existing_existing_pairs["facility_id_2"]
].copy()

existing_existing_pairs["distance_miles"] = haversine_miles(
    existing_existing_pairs["latitude_1"],
    existing_existing_pairs["longitude_1"],
    existing_existing_pairs["latitude_2"],
    existing_existing_pairs["longitude_2"],
)

existing_existing_too_close = existing_existing_pairs[
    existing_existing_pairs["distance_miles"] < MIN_DIST
][["facility_id_1", "facility_id_2", "zipcode", "distance_miles"]].drop_duplicates()

print("Existing-existing close pairs (diagnostic only):", existing_existing_too_close.shape)
display(existing_existing_too_close.head())

Existing-existing close pairs (diagnostic only): (7644, 4)


,facility_id_1,facility_id_2,zipcode,distance_miles
5,51349,609919,10474,0.027891
11,51349,633751,10474,0.027891
13,51349,601505,10474,0.042375
411,274852,385789,10460,0.055933
467,274852,743579,10460,0.037859


In [15]:
# Blocked candidates and feasible candidate-size options
blocked_candidates = (
    candidate_existing_conflicts[["candidate_id"]]
    .drop_duplicates()
    .assign(blocked_by_existing=1)
)

candidate_geo_realistic = candidate_geo.merge(
    blocked_candidates,
    on="candidate_id",
    how="left",
)
candidate_geo_realistic["blocked_by_existing"] = candidate_geo_realistic["blocked_by_existing"].fillna(0).astype(int)

candidate_build_options_realistic = candidate_build_options_realistic.merge(
    candidate_geo_realistic[["candidate_id", "blocked_by_existing"]],
    on="candidate_id",
    how="left",
)
candidate_build_options_realistic["blocked_by_existing"] = (
    candidate_build_options_realistic["blocked_by_existing"].fillna(0).astype(int)
)

candidate_build_options_feasible = candidate_build_options_realistic[
    candidate_build_options_realistic["blocked_by_existing"] == 0
].copy()

print(candidate_geo_realistic["blocked_by_existing"].value_counts(dropna=False))
print("All candidate-size options:", candidate_build_options_realistic.shape)
print("Feasible candidate-size options:", candidate_build_options_feasible.shape)
display(candidate_geo_realistic.head())

blocked_by_existing
0    28522
1     2578
Name: count, dtype: int64
All candidate-size options: (93300, 10)
Feasible candidate-size options: (85566, 10)


,candidate_id,zipcode,latitude,longitude,geo_missing_flag,blocked_by_existing
0,1,10001,40.741893,-74.000140,False,0
1,2,10001,40.752007,-74.005436,False,0
2,3,10001,40.750545,-73.997147,False,0
3,4,10001,40.744080,-74.001932,False,0
4,5,10001,40.748690,-73.999341,False,1


## Summary and exports

The final export block writes all realistic parameter tables to disk and records a compact summary of the feasible expansion and siting environment.


In [16]:
# Summary table for the realistic parameter build
realistic_parameter_summary = pd.DataFrame({
    "metric": [
        "n_zipcodes",
        "n_existing_facilities_expandable",
        "citywide_existing_total_capacity",
        "citywide_existing_under5_capacity",
        "citywide_realistic_expansion_capacity",
        "n_candidate_sites",
        "n_blocked_candidate_sites",
        "n_feasible_candidate_sites",
        "n_candidate_candidate_conflicts",
        "n_candidate_existing_conflicts",
        "n_existing_existing_too_close_diagnostic",
        "n_zero_child_zipcodes",
        "n_zero_child_req_total_fixed",
        "n_existing_facilities_missing_coords",
    ],
    "value": [
        zipcode_demand_supply_realistic["zipcode"].nunique(),
        facility_expansion_params_realistic["facility_id"].nunique(),
        facility_expansion_params_realistic["n_f"].sum(),
        facility_expansion_params_realistic["b_f"].sum(),
        facility_expansion_params_realistic["U_f_realistic"].sum(),
        candidate_geo_realistic["candidate_id"].nunique(),
        candidate_geo_realistic["blocked_by_existing"].sum(),
        (candidate_geo_realistic["blocked_by_existing"] == 0).sum(),
        len(candidate_candidate_conflicts),
        len(candidate_existing_conflicts),
        len(existing_existing_too_close),
        int(zero_child_mask.sum()),
        zero_child_req_total_fixed,
        int(facility_expansion_params_realistic["latitude"].isna().sum()),
    ],
})

display(realistic_parameter_summary)

,metric,value
0,n_zipcodes,311
1,n_existing_facilities_expandable,7712
2,citywide_existing_total_capacity,317501
3,citywide_existing_under5_capacity,69153
4,citywide_realistic_expansion_capacity,61197
5,n_candidate_sites,31100
6,n_blocked_candidate_sites,2578
7,n_feasible_candidate_sites,28522
8,n_candidate_candidate_conflicts,11455
9,n_candidate_existing_conflicts,4730


In [17]:
# Save assumptions tables
REALISTIC_ASSUMPTIONS = {
    "distance_threshold_miles": 0.06,
    "distance_enforced_within_zipcode_only": True,
    "existing_existing_conflicts_are_diagnostic_only": True,
    "candidate_existing_conflict_blocks_candidate": True,
    "use_under5_capacity_current_for_existing_capacity": True,
    "new_build_costs_and_capacities_reused_from_ideal_model": True,
    "realistic_expansion_cap_implemented_as_floor_20pct_total": True,
    "expansion_block_caps_from_cumulative_floors": True,
    "zero_child_zipcodes_have_zero_total_requirement": True,
    "canonical_part2_cost_model": "additive_blocks",
}

with open(REALISTIC_DIR / "assumptions_realistic.json", "w", encoding="utf-8") as f:
    json.dump(REALISTIC_ASSUMPTIONS, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "assumption": list(REALISTIC_ASSUMPTIONS.keys()),
    "value": list(REALISTIC_ASSUMPTIONS.values()),
}).to_csv(REALISTIC_DIR / "assumptions_realistic.csv", index=False)

print("Saved assumptions_realistic.json and assumptions_realistic.csv")

Saved assumptions_realistic.json and assumptions_realistic.csv


In [18]:
# Export all realistic parameter files
facility_expansion_params_realistic.to_csv(
    REALISTIC_DIR / "facility_expansion_params_realistic.csv",
    index=False,
)
zipcode_demand_supply_realistic.to_csv(
    REALISTIC_DIR / "zipcode_demand_supply_realistic.csv",
    index=False,
)
candidate_geo_realistic.to_csv(
    REALISTIC_DIR / "candidate_sites_realistic.csv",
    index=False,
)
candidate_build_options_feasible.to_csv(
    REALISTIC_DIR / "candidate_build_options_realistic.csv",
    index=False,
)
candidate_existing_conflicts.to_csv(
    REALISTIC_DIR / "candidate_existing_conflicts.csv",
    index=False,
)
candidate_candidate_conflicts.to_csv(
    REALISTIC_DIR / "candidate_candidate_conflicts.csv",
    index=False,
)
existing_existing_too_close.to_csv(
    REALISTIC_DIR / "existing_existing_too_close_diagnostic.csv",
    index=False,
)
realistic_parameter_summary.to_csv(
    REALISTIC_DIR / "realistic_parameter_summary.csv",
    index=False,
)

print("Realistic parameter files saved to:", REALISTIC_DIR)

Realistic parameter files saved to: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/realistic


In [19]:
# Final QA preview
print("zipcode_demand_supply_realistic:", zipcode_demand_supply_realistic.shape)
print("facility_expansion_params_realistic:", facility_expansion_params_realistic.shape)
print("candidate_sites_realistic:", candidate_geo_realistic.shape)
print("candidate_build_options_realistic:", candidate_build_options_feasible.shape)
print("candidate_existing_conflicts:", candidate_existing_conflicts.shape)
print("candidate_candidate_conflicts:", candidate_candidate_conflicts.shape)
print("existing_existing_too_close_diagnostic:", existing_existing_too_close.shape)

display(realistic_parameter_summary)

zipcode_demand_supply_realistic: (311, 23)
facility_expansion_params_realistic: (7712, 18)
candidate_sites_realistic: (31100, 6)
candidate_build_options_realistic: (85566, 10)
candidate_existing_conflicts: (4730, 4)
candidate_candidate_conflicts: (11455, 4)
existing_existing_too_close_diagnostic: (7644, 4)


,metric,value
0,n_zipcodes,311
1,n_existing_facilities_expandable,7712
2,citywide_existing_total_capacity,317501
3,citywide_existing_under5_capacity,69153
4,citywide_realistic_expansion_capacity,61197
5,n_candidate_sites,31100
6,n_blocked_candidate_sites,2578
7,n_feasible_candidate_sites,28522
8,n_candidate_candidate_conflicts,11455
9,n_candidate_existing_conflicts,4730
